In [187]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Generisanje simuliranih podataka
np.random.seed(42)
brands = ['Apple', 'Dell', 'HP', 'Lenovo', 'Asus', 'Acer', 'MSI']
cpus = ['Intel Core i7 12700H', 'AMD Ryzen 5 5600U', 'Intel Core i5 1135G7', 'Apple M2', 'Intel Celeron N4020']
gpus = ['NVIDIA RTX 3060', 'Intel Iris Xe', 'AMD Radeon', 'Integrated', 'NVIDIA RTX 4070']

data = {
    'laptop_id': range(100),
    'full_name': [f"{np.random.choice(brands)} {np.random.choice(['Pro', 'Air', 'Gaming', 'Home'])}" for _ in range(100)],
    'cpu_info': [np.random.choice(cpus) for _ in range(100)],
    'gpu_info': [np.random.choice(gpus) for _ in range(100)],
    'ram_storage': [f"{np.random.choice([8, 16, 32])}GB | {np.random.choice([256, 512, 1024])}GB SSD" for _ in range(100)],
    'price_raw': [f"${np.random.randint(400, 2500)}.00" for _ in range(100)],
    'weight_kg': [f"{np.random.uniform(1.2, 3.5):.2f}kg" for _ in range(100)]
}

df = pd.DataFrame(data)

# Ubacivanje malo "haosa"
df.iloc[5:10, 5] = np.nan  # Nedostajuće cene
df.loc[95:, 'full_name'] = df.iloc[0, 1]  # Namerni duplikati

### Samo "laptop_id" nije object, imamo 5 NaN vrednosti u "price_raw" i nemamo duplikate

In [188]:
# print(df.dtypes)
# print(df[df.duplicated()])
# print(df.duplicated().sum())
# print(df.isnull().sum())
# print(df.isna().sum())

### 1. Inicijalno ciscenje - duplikati, nan, convert

In [189]:
df.drop_duplicates(keep='first', inplace=True, ignore_index=True) # nepotrebno

df.dropna(axis=0, inplace=True, ignore_index=True)

df["price"] = df["price_raw"].str.replace("$", "").str.replace(".00", "")
df["price"] = pd.to_numeric(df["price"], errors="coerce")

df["weight"] = df["weight_kg"].str.replace("kg", "")
df["weight"] = pd.to_numeric(df["weight"], errors="coerce")

df.drop(axis=1, columns=["price_raw", "weight_kg"], inplace=True)

### 2. Feature Extraction

In [190]:
# print(df["ram_storage"].head())

df[['ram', 'storage']] = df['ram_storage'].str.split('|', expand=True)

df["ram"] = df["ram"].str.strip().str.replace("GB", "").astype(int)
df["storage"] = df["storage"].str.strip().str.replace("GB SSD", "").astype(int)

# print(df["ram"])
# print(df["storage"])

In [191]:
# print(df["cpu_info"])

def feature_extract_cpu(text):
    parts = str(text).split(" ")
    try:
        return parts[0]
    except [ValueError, IndexError]:
        return np.nan
    
df["cpu_brand"] = df["cpu_info"].apply(feature_extract_cpu).str.strip()
# print(df["cpu_brand"])

In [192]:
# print(df["gpu_info"])

def feature_extract_gpu(text):
    if "NVIDIA" in str(text):
        return 1
    else:
        return 0
    
df["gaming_gpu"] = df["gpu_info"].apply(feature_extract_gpu)
# print(df["gaming_gpu"])

In [193]:
df.drop(axis=1, columns=["cpu_info", "gpu_info", "ram_storage"], inplace=True)

### 3. Agregacija

In [194]:
avg_price_cpu = df.groupby("cpu_brand")["price"].mean()
# print(avg_price_cpu)

In [195]:
df[['brand', 'type']] = df['full_name'].str.split(" ", expand=True)
    
df["brand"] = df["brand"].str.strip()
df["type"] = df["type"].str.strip()


gaming_laptops_brand = df.groupby("brand")["gaming_gpu"].sum()
# print(gaming_laptops_brand.sort_values(ascending=False)) # 39 are gaming (have gaming gpu)

gaming_laptops_type = df["type"].value_counts()
# print(gaming_laptops_type) # 23 are gaming (gaming in their name)
# print(df.head())


### 4. Vizuelizacija

In [196]:
fig = px.box(df, 
             x = "price", 
             y = "cpu_brand",
             color = "cpu_brand",
             color_discrete_map={"Intel": "#00D9FF", "Apple": "#B3FF00", "AMD": "#FF0000"},
             points="all",
             template="plotly_dark")

fig.update_xaxes(title=dict(text="Prices (USD)", font=dict(size=16, color="#009DFF")))
fig.update_yaxes(title=dict(text="CPU Brands", font=dict(size=16, color="#009DFF")))

fig.update_layout(title=dict(text="Price comparison of different CPU brands", font=dict(size=20, color="#009DFF"), xanchor="center", x = 0.5, y = 0.95),
                  showlegend=False)

fig.update_traces(hovertemplate="<b>Brand</b>: %{y}<br><b>Price ($)</b>: %{x}")


In [197]:
fig_bubble = px.scatter(df, 
                  x="weight", 
                  y="price", 
                  size="ram",  # Veličina mehurića zavisi od RAM-a
                  color="cpu_brand", 
                  hover_name="brand",
                  template="plotly_dark",
                  title="Weight vs Price (Size = RAM)")

fig_bubble.show()